In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    average_precision_score,
)
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import numpy as np

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
df_raw = pd.read_csv("accepted_2007_to_2018Q4.csv", usecols=["recoveries"])

In [13]:
# ── 인덱스 범위 비교 ──────────────────────────
print(f"df_raw 행 수:   {len(df_raw)}")
print(f"test_df 최대 인덱스: {test_df.index.max()}")
print(f"test_df 최소 인덱스: {test_df.index.min()}")

# test_df 인덱스가 df_raw 범위 안에 있는지 확인
print(f"df_raw로 커버 가능: {test_df.index.max() < len(df_raw)}")

df_raw 행 수:   2260701
test_df 최대 인덱스: 1277798
test_df 최소 인덱스: 3
df_raw로 커버 가능: True


In [7]:
import joblib
import pickle

# ── joblib 불러오기 ───────────────────────────
best_model = joblib.load("model/loan_default_model.joblib")
print(type(best_model))  # Pipeline인지 CatBoost 단독인지 확인

# ── pkl 불러오기 ──────────────────────────────
with open("model/catboost_model.pkl", "rb") as f:
    cat_model = pickle.load(f)
print(type(cat_model))

<class 'dict'>
<class 'catboost.core.CatBoostClassifier'>


In [8]:
# dict 키 확인
print(best_model.keys())

# 각 키별 타입 확인
for k, v in best_model.items():
    print(f"{k}: {type(v)}")

dict_keys(['model', 'threshold', 'features'])
model: <class 'catboost.core.CatBoostClassifier'>
threshold: <class 'float'>
features: <class 'list'>


In [30]:
use_col = ['loan_amnt', 'term', 'int_rate', 'installment', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'issue_d', 'purpose',
    'dti', 'earliest_cr_line', 'mths_since_last_delinq', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'mths_since_last_major_derog',
    'mths_since_rcnt_il', 'total_rev_hi_lim', 'acc_open_past_24mths',
    'avg_cur_bal', 'bc_open_to_buy', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mths_since_recent_bc',
       'mths_since_recent_bc_dlq', 'mths_since_recent_inq',
       'mths_since_recent_revol_delinq', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_il_tl', 'num_rev_tl_bal_gt_0', 'pct_tl_nvr_dlq', 'tot_hi_cred_lim',
       'total_bc_limit', 'issue_year', 'issue_month',
       'installment_to_income', 'loan_to_income', 'revol_bal_to_income',
       'fico_mid', 'mths_since_last_delinq_flag',
       'mths_since_last_major_derog_flag',
       'mths_since_recent_revol_delinq_flag', 'mths_since_recent_bc_dlq_flag',
       'mths_since_recent_inq_flag', 'mths_since_recent_bc_flag', 'target']

In [37]:
# ── 컬럼별 타입 확인 ──────────────────────────
print(X_test_input.dtypes)
print("\n")
# ── object 타입 컬럼 확인 ─────────────────────
obj_cols = X_test_input.select_dtypes(include="object").columns.tolist()
print(f"object 타입 컬럼: {obj_cols}")

# ── object 컬럼 샘플값 확인 ───────────────────
for col in obj_cols:
    print(f"\n{col} 샘플값:")
    print(X_test_input[col].value_counts().head())

loan_amnt                              float64
term                                       str
int_rate                               float64
installment                            float64
sub_grade                                  str
emp_length                                 str
home_ownership                             str
annual_inc                             float64
issue_d                                    str
purpose                                    str
dti                                    float64
earliest_cr_line                           str
mths_since_last_delinq                 float64
pub_rec                                float64
revol_bal                              float64
revol_util                             float64
total_acc                              float64
mths_since_last_major_derog            float64
mths_since_rcnt_il                     float64
total_rev_hi_lim                       float64
acc_open_past_24mths                   float64
avg_cur_bal  

C:\Users\Cho\AppData\Local\Temp\ipykernel_13104\4081377777.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = X_test_input.select_dtypes(include="object").columns.tolist()


issue_d
2016-03-01    9779
2015-10-01    8662
2015-07-01    8135
2015-12-01    7746
2014-10-01    7163
Name: count, dtype: int64

purpose 샘플값:
purpose
debt_consolidation    149401
credit_card            56914
home_improvement       16569
other                  14251
major_purchase          5321
Name: count, dtype: int64

earliest_cr_line 샘플값:
earliest_cr_line
Aug-2001    1812
Aug-2002    1753
Sep-2003    1706
Oct-2001    1695
Aug-2003    1684
Name: count, dtype: int64


In [35]:
print(f"4번째 컬럼: {X_test_input.columns[3]}")
print(X_test_input.iloc[:, 3].dtype)
print(X_test_input.iloc[:, 3].head())

4번째 컬럼: installment
float64
211797    610.62
820519    284.30
974578    460.56
698374    622.04
596959    122.92
Name: installment, dtype: float64


In [41]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

use_col_with_target = use_col                                           # df2용 (target 포함)
use_col_features    = [col for col in use_col if col != "target"]      # X_test용 (target 제외)


# ── 데이터 불러오기 ───────────────────────────
df = pd.read_csv("lending_club_preprocessed3.csv")
df2 = df[use_col_with_target].copy()  # use_col 필터링

y = df2["target"]
X = df2.drop(columns=["target"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [42]:
# 파생변수 결측값 채우기
new_cols = ['installment_to_income', 'loan_to_income', 'revol_bal_to_income']
for col in new_cols:
    train_median = X_train[col].median()                    # train으로만 계산
    X_train[col] = X_train[col].fillna(train_median)       # train에 적용
    X_test[col] = X_test[col].fillna(train_median)         # test에도 train median 사용

# ★train/test 이후 전처리
# dti "sub_grade" 사용해야함 
X_train["dti"] = X_train.groupby("sub_grade")["dti"].transform(lambda x: x.fillna(x.median())
)
train_dit_medi = X_train.groupby("sub_grade")["dti"].median()
X_test["dti"] = X_test["dti"].fillna(X_test["sub_grade"].map(train_dit_medi))

# 신용 조회, 
cols = [
'mths_since_last_delinq',
'mths_since_last_major_derog',
'mths_since_recent_revol_delinq',
'mths_since_recent_bc_dlq',
'mths_since_recent_inq',
'mths_since_recent_bc',
'mths_since_rcnt_il',
'mo_sin_old_il_acct'
]
for col in cols:
    max_val = X_train[col].max()
    X_train[col] = X_train[col].fillna(max_val + 1)
    X_test[col] = X_test[col].fillna(max_val + 1)
    
# pct_tl_nvr_dlq 연체 이력이 한 번도 없는 계좌 비율 (%)
## 중앙값으로 채우기 
medi_dlq = X_train["pct_tl_nvr_dlq"].median()
X_train["pct_tl_nvr_dlq"] = X_train["pct_tl_nvr_dlq"].fillna(medi_dlq)
X_test["pct_tl_nvr_dlq"] = X_test["pct_tl_nvr_dlq"].fillna(medi_dlq)

In [43]:
# ── 모델 불러오기 ─────────────────────────────
best_model = joblib.load("model/loan_default_model2.joblib")
model     = best_model["model"]
threshold = best_model["threshold"]
features  = best_model["features"]

print(f"threshold: {threshold}")
print(f"features 수: {len(features)}")

# ── features 누락 확인 ────────────────────────
missing = [f for f in features if f not in X_test.columns]
print(f"누락 features: {missing}")  # 반드시 [] 나와야 함


# ── 예측 ──────────────────────────────────────
X_test_input = X_test[features]  # 모델에서 X값 
y_prob = model.predict_proba(X_test_input)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

# ── TP/TN/FP/FN 매핑 ──────────────────────────
test_df = X_test_input.copy()
test_df["target"] = y_test.values
test_df["y_pred"] = y_pred
test_df["y_prob"] = y_prob

def map_confusion(actual, pred):
    if actual == 1 and pred == 1:
        return "TP"
    elif actual == 0 and pred == 0:
        return "TN"
    elif actual == 0 and pred == 1:
        return "FP"
    else:
        return "FN"

test_df["confusion"] = test_df.apply(
    lambda row: map_confusion(row["target"], row["y_pred"]), axis=1
)

print(test_df["confusion"].value_counts())

threshold: 0.525
features 수: 39
누락 features: []
confusion
TN    143436
FP     60469
TP     32911
FN     18744
Name: count, dtype: int64


In [44]:
print("X_test_input 결측값 수:")
print(X_test_input.isnull().sum()[X_test_input.isnull().sum() > 0])

X_test_input 결측값 수:
Series([], dtype: int64)


In [46]:
# ── loan_amnt, int_rate, term 원본에서 가져오기 ─
test_df["loan_amnt"] = df.loc[test_df.index, "loan_amnt"]
test_df["int_rate"]  = df.loc[test_df.index, "int_rate"]
test_df["term"]      = df.loc[test_df.index, "term"]

# ── term 숫자로 변환 ──────────────────────────
test_df["term_num"] = test_df["term"].str.extract(r"(\d+)").astype(int)

# ── interest 재정의 ───────────────────────────
test_df["interest"] = (
    test_df["loan_amnt"] * (test_df["int_rate"] / 100 / 12) * test_df["term_num"]
)

# ────────────────────────────────────────────────
# 모델 미사용 손실 (A)
# 전부 승인 → 부도(TP+FN) 원금 손실
# ────────────────────────────────────────────────
tp_loss = test_df[test_df["confusion"] == "TP"]["loan_amnt"].sum()
fn_loss = test_df[test_df["confusion"] == "FN"]["loan_amnt"].sum()
A_loss  = tp_loss + fn_loss

# ────────────────────────────────────────────────
# 모델 사용 손실 (B)
# FN 원금 손실 + FP 이자 기회비용
# ────────────────────────────────────────────────
fp_opportunity = test_df[test_df["confusion"] == "FP"]["interest"].sum()
B_loss = fn_loss + fp_opportunity

# ── 손실 감소 효과 ────────────────────────────
loss_reduction     = A_loss - B_loss
loss_reduction_pct = (loss_reduction / A_loss) * 100

print("=" * 60)
print("📊 모델 사용 전후 손실 비교")
print("=" * 60)
print(f"[A] 모델 미사용 시 총 손실 (전부 승인)")
print(f"    ├ TP 원금 손실 (차단 가능했던):  -${tp_loss:>15,.2f}")
print(f"    ├ FN 원금 손실 (놓친 부도):      -${fn_loss:>15,.2f}")
print(f"    └ 총 손실:                       -${A_loss:>15,.2f}")
print(f"\n[B] 모델 사용 시 총 손실")
print(f"    ├ FN 원금 손실 (놓친 부도):      -${fn_loss:>15,.2f}")
print(f"    ├ FP 기회비용 (정상 거절):       -${fp_opportunity:>15,.2f}")
print(f"    └ 총 손실:                       -${B_loss:>15,.2f}")
print(f"\n손실 감소액 (A - B):                  ${loss_reduction:>15,.2f}")
print(f"손실 감소율:                          {loss_reduction_pct:>14.2f}%")
print("=" * 60)

# ── confusion별 상세 ──────────────────────────
print("\n📋 confusion별 상세")
detail = test_df.groupby("confusion").agg(
    건수=("loan_amnt", "count"),
    원금합계=("loan_amnt", "sum"),
    이자합계=("interest", "sum")
).round(2)
print(detail)

### 계산 구조

# A (미사용 손실) = TP 원금 + FN 원금
#                   (부도 전부 승인했을 때 손실)

# B (사용 손실)  = FN 원금 + FP 기회비용
#                   (모델 적용 후 남은 손실)

# 손실 감소액    = A - B  → TP가 차단한 원금 - FP 기회비용
# 손실 감소율    = (A - B) / A × 100

📊 모델 사용 전후 손실 비교
[A] 모델 미사용 시 총 손실 (전부 승인)
    ├ TP 원금 손실 (차단 가능했던):  -$ 557,565,450.00
    ├ FN 원금 손실 (놓친 부도):      -$ 254,222,250.00
    └ 총 손실:                       -$ 811,787,700.00

[B] 모델 사용 시 총 손실
    ├ FN 원금 손실 (놓친 부도):      -$ 254,222,250.00
    ├ FP 기회비용 (정상 거절):       -$ 701,148,225.32
    └ 총 손실:                       -$ 955,370,475.32

손실 감소액 (A - B):                  $-143,582,775.32
손실 감소율:                                  -17.69%

📋 confusion별 상세
               건수          원금합계          이자합계
confusion                                    
FN          18744  2.542222e+08  1.074315e+08
FP          60469  9.768140e+08  7.011482e+08
TN         143436  1.923934e+09  7.002501e+08
TP          32911  5.575654e+08  4.404682e+08


In [48]:
# ── threshold별 손실 시뮬레이션 ──────────────
results = []

for thr in np.arange(0.1, 0.9, 0.05):
    y_pred_thr = (y_prob >= thr).astype(int)

    temp_df = test_df.copy()
    temp_df["y_pred"] = y_pred_thr
    temp_df["confusion"] = temp_df.apply(
        lambda row: map_confusion(row["target"], row["y_pred"]), axis=1
    )

    tp_l = temp_df[temp_df["confusion"] == "TP"]["loan_amnt"].sum()
    fn_l = temp_df[temp_df["confusion"] == "FN"]["loan_amnt"].sum()
    fp_o = temp_df[temp_df["confusion"] == "FP"]["interest"].sum()

    A = tp_l + fn_l
    B = fn_l + fp_o
    reduction = A - B

    counts = temp_df["confusion"].value_counts()
    results.append({
        "threshold":  round(thr, 2),
        "손실감소액": round(reduction, 2),
        "손실감소율": round((reduction / A) * 100, 2) if A > 0 else 0,
        "TN": counts.get("TN", 0),
        "FP": counts.get("FP", 0),
        "TP": counts.get("TP", 0),
        "FN": counts.get("FN", 0),
    })

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

# ── 손실 감소액 최대 threshold ────────────────
best = result_df.loc[result_df["손실감소액"].idxmax()]
print(f"\n✅ 최적 threshold: {best['threshold']}")
print(f"   손실 감소액:     ${best['손실감소액']:,.2f}")
print(f"   손실 감소율:     {best['손실감소율']}%")
print(f"   TN: {int(best['TN']):,}  FP: {int(best['FP']):,}  TP: {int(best['TP']):,}  FN: {int(best['FN']):,}")

 threshold         손실감소액  손실감소율     TN     FP    TP    FN
      0.10 -5.683154e+08 -70.01   8966 194939 51501   154
      0.15 -5.406384e+08 -66.60  19667 184238 51087   568
      0.20 -5.047752e+08 -62.18  32606 171299 50374  1281
      0.25 -4.610090e+08 -56.79  47875 156030 49250  2405
      0.30 -4.109366e+08 -50.62  64470 139435 47658  3997
      0.35 -3.557490e+08 -43.82  82129 121776 45489  6166
      0.40 -2.981230e+08 -36.72 100183 103722 42607  9048
      0.45 -2.356722e+08 -29.03 118124  85781 39186 12469
      0.50 -1.752972e+08 -21.59 135295  68610 35076 16579
      0.55 -1.152046e+08 -14.19 151241  52664 30530 21125
      0.60 -6.103156e+07  -7.52 165657  38248 25562 26093
      0.65 -1.456370e+07  -1.79 177872  26033 20459 31196
      0.70  1.756932e+07   2.16 187649  16256 15259 36396
      0.75  3.409712e+07   4.20 195112   8793 10177 41478
      0.80  3.056408e+07   3.77 199981   3924  5667 45988
      0.85  1.987410e+07   2.45 202800   1105  2264 49391

✅ 최적 threshol